In [2]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
from tensorflow import keras
from tqdm import tqdm
import numpy as np
from datetime import datetime
import logging
import json
import matplotlib.pyplot as plt

# ===================================================================
# MAIN EXECUTION
# ===================================================================
if __name__ == "__main__":

    # 1. KONFIGURASI PATH
    BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    
    # Path Data Test (Blind Test)
    TEST_DATA_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json"
    
    # Path KDE Indonesia
    EMB_DIR_INDO = "/Volumes/Extreme SSD/unduhan_waveform_geofon/output/indonesia_domain_embeddings_3c_le"

    INPUT_WIN = 7 
    SAMPLING_RATE = 100
    num_points = int(INPUT_WIN * SAMPLING_RATE)
    
    # 2. PERSIAPAN OUTPUT
    SAVE_BASE = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_5/output'
    now = datetime.now()
    time_str = now.strftime("%d%H%M%S")
    save_dir = os.path.join(SAVE_BASE, f"Eval_Error_Analysis_{time_str}")
    if not os.path.exists(save_dir): 
        os.makedirs(save_dir)

    # ===================================================================
    # 3. MEMUAT MODEL & KDE
    # ===================================================================
    print("[INFO] Memuat Model dan Membangun KDE...")
    embedding_model = keras.models.load_model(filepath=MODEL_PATH)

    emb_Z = dataset.load_embedding_data(EMB_DIR_INDO, "Embedding data, Z.json")
    emb_N = dataset.load_embedding_data(EMB_DIR_INDO, "Embedding data, N.json")
    emb_E = dataset.load_embedding_data(EMB_DIR_INDO, "Embedding data, E.json")
    kde_indo = utils.embedding_PDFs_3D(emb_Z, emb_N, emb_E, source_list=['noise', 'le'])

    # ===================================================================
    # 4. INFERENSI & PENCATATAN ERROR
    # ===================================================================
    print(f"[INFO] Memuat Data Test: {TEST_DATA_PATH}")
    test_data = dataset.load_json_data(TEST_DATA_PATH)
    keys_list = list(test_data.keys())
    
    total_true, total_pred = [], []
    
    # Wadah pencatat ID error
    misclassified_records = {
        "False_Positives": [], 
        "False_Negatives": []  
    }
    
    for i in tqdm(range(len(keys_list)), desc="Evaluasi & Ekstraksi Error"):
        record_key = keys_list[i]
        record = test_data[record_key]
        
        try:
            # Ambil sinyal
            Z_n, N_n, E_n = record["Z_noise"][-num_points:], record["N_noise"][-num_points:], record["E_noise"][-num_points:]
            Z_s, N_s, E_s = record["Z"][:num_points], record["N"][:num_points], record["E"][:num_points]

            # Proyeksikan ke ruang laten
            _in_Zn, _in_Nn, _in_En = utils.latent_codes_1D(Z_n, embedding_model), utils.latent_codes_1D(N_n, embedding_model), utils.latent_codes_1D(E_n, embedding_model)
            _in_Zs, _in_Ns, _in_Es = utils.latent_codes_1D(Z_s, embedding_model), utils.latent_codes_1D(N_s, embedding_model), utils.latent_codes_1D(E_s, embedding_model)

            # Uji jarak ke KDE
            emb_n_3c = np.array([_in_En, _in_Nn, _in_Zn]).reshape(1,-1)
            p_n, _, _ = utils.infer_3C_PDFs(emb_n_3c, kde_indo, "Kernel")
            
            emb_s_3c = np.array([_in_Es, _in_Ns, _in_Zs]).reshape(1,-1)
            p_s, _, _ = utils.infer_3C_PDFs(emb_s_3c, kde_indo, "Kernel")

            # Prediksi
            pred_n = 1 if p_n >= 1 else 0  # Harus 0
            pred_s = 1 if p_s >= 1 else 0  # Harus 1

            # Pencatatan ID Error
            if pred_n == 1:
                misclassified_records["False_Positives"].append(record_key)
            if pred_s == 0:
                misclassified_records["False_Negatives"].append(record_key)

            total_true.extend([0, 1]) 
            total_pred.extend([pred_n, pred_s])
            
        except Exception as e:
            continue
            
    # ===================================================================
    # 5. SIMPAN HASIL
    # ===================================================================
    # Simpan JSON ID Error
    error_log_path = os.path.join(save_dir, "Misclassified_IDs_INDONESIA_TEST.json")
    with open(error_log_path, 'w') as f:
        json.dump(misclassified_records, f, indent=4)
        
    print(f"\n✅ EKSTRAKSI SELESAI!")
    print(f"Total False Positives dicatat: {len(misclassified_records['False_Positives'])}")
    print(f"Total False Negatives dicatat: {len(misclassified_records['False_Negatives'])}")
    print(f"File JSON Error tersimpan di: {error_log_path}")

[INFO] Memuat Model dan Membangun KDE...


2026-08-06 10:43:41.136796: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2026-08-06 10:43:41.136820: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2026-08-06 10:43:41.136825: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
2026-08-06 10:43:41.137027: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-06 10:43:41.137384: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


[INFO] Memuat Data Test: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json


Evaluasi & Ekstraksi Error: 100%|██████████| 1032/1032 [00:24<00:00, 42.58it/s]


✅ EKSTRAKSI SELESAI!
Total False Positives dicatat: 288
Total False Negatives dicatat: 32
File JSON Error tersimpan di: /Volumes/Extreme SSD/json_indonesia_juli_sesi_5/output/Eval_Error_Analysis_06104341/Misclassified_IDs_INDONESIA_TEST.json


In [8]:
# -*- coding: utf-8 -*-
import os
import json
import random
import matplotlib.pyplot as plt

# ==========================================
# 1. KONFIGURASI PATH (ABSOLUT)
# ==========================================
# Path ke dataset pengujian
DATA_JSON_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json"

# Path ke file JSON berisi ID yang salah tebak
ERROR_IDS_PATH = "/Volumes/Extreme SSD/json_indonesia_juli_sesi_5/output/Eval_Error_Analysis_06104341/Misclassified_IDs_INDONESIA_TEST.json"

# Memaksa folder output dibuat persis di dalam folder Eval_Error_Analysis di SSD Anda
BASE_OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/outpu_error_sta_lta'
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "error_analysis_plots")

# Jumlah sampel yang ingin diekstrak dan diplot secara acak per kategori
NUM_SAMPLES_TO_PLOT = 5  

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 2. FUNGSI PLOTTING 3C (ENGLISH LABELS)
# ==========================================
def plot_3c_waveform(data_record, record_id, error_type, save_dir):
    input_size = 700
    
    if error_type == "False_Positive":
        sig_Z = data_record["Z_noise"][-input_size:]
        sig_N = data_record["N_noise"][-input_size:]
        sig_E = data_record["E_noise"][-input_size:]
        title_text = f"FALSE POSITIVE (Model Pred: Earthquake | STA/LTA Label: Noise)\nRecord ID: {record_id}"
        line_color = '#e74c3c' 
        
    elif error_type == "False_Negative":
        sig_Z = data_record["Z"][:input_size]
        sig_N = data_record["N"][:input_size]
        sig_E = data_record["E"][:input_size]
        title_text = f"FALSE NEGATIVE (Model Pred: Noise | STA/LTA Label: Earthquake)\nRecord ID: {record_id}"
        line_color = '#3498db' 
    else:
        return

    fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    fig.suptitle(title_text, fontsize=12, fontweight='bold')
    
    # Komponen Z
    axs[0].plot(sig_Z, color='k', linewidth=1.2, label='Z Component')
    axs[0].set_ylabel('Amplitude')
    axs[0].legend(loc='upper right')
    axs[0].grid(True, linestyle='--', alpha=0.6)
    
    # Komponen N
    axs[1].plot(sig_N, color=line_color, linewidth=1, label='N Component')
    axs[1].set_ylabel('Amplitude')
    axs[1].legend(loc='upper right')
    axs[1].grid(True, linestyle='--', alpha=0.6)
    
    # Komponen E
    axs[2].plot(sig_E, color=line_color, linewidth=1, label='E Component')
    axs[2].set_ylabel('Amplitude')
    axs[2].set_xlabel('Time (Samples @ 100Hz)')
    axs[2].legend(loc='upper right')
    axs[2].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9) 
    
    file_name = f"{error_type}_{record_id}.jpg"
    plt.savefig(os.path.join(save_dir, file_name), dpi=300)
    plt.close()
    print(f"  [+] Saved: {file_name}")

# ==========================================
# 3. EKSEKUSI UTAMA
# ==========================================
def main():
    print(f"Reading main dataset...")
    with open(DATA_JSON_PATH, "r") as f:
        test_data = json.load(f)

    print(f"Reading misclassified IDs log...")
    with open(ERROR_IDS_PATH, "r") as f:
        error_ids = json.load(f)

    fp_list = error_ids.get("False_Positives", [])
    fn_list = error_ids.get("False_Negatives", [])

    sample_fp = random.sample(fp_list, min(NUM_SAMPLES_TO_PLOT, len(fp_list)))
    sample_fn = random.sample(fn_list, min(NUM_SAMPLES_TO_PLOT, len(fn_list)))

    print("\n--- Plotting False Positives ---")
    for rec_id in sample_fp:
        if rec_id in test_data:
            plot_3c_waveform(test_data[rec_id], rec_id, "False_Positive", OUTPUT_DIR)

    print("\n--- Plotting False Negatives ---")
    for rec_id in sample_fn:
        if rec_id in test_data:
            plot_3c_waveform(test_data[rec_id], rec_id, "False_Negative", OUTPUT_DIR)
            
    print(f"\n✅ Finished! Open this directory in Finder to view the plots:")
    print(f"➡️  {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Reading main dataset...
Reading misclassified IDs log...

--- Plotting False Positives ---
  [+] Saved: False_Positive_GE_BKNI_20091228_134028.jpg
  [+] Saved: False_Positive_GE_TNTI_20070307_060541.jpg
  [+] Saved: False_Positive_GE_TOLI_20100401_144539.jpg
  [+] Saved: False_Positive_GE_GSI_20080404_164332.jpg
  [+] Saved: False_Positive_GE_UGM_20050331_213646.jpg

--- Plotting False Negatives ---
  [+] Saved: False_Negative_GE_MMRI_20080603_210346.jpg
  [+] Saved: False_Negative_GE_GSI_20100407_194505.jpg
  [+] Saved: False_Negative_XN_KRI_20060706_043857.jpg
  [+] Saved: False_Negative_GE_GSI_20070501_231303.jpg
  [+] Saved: False_Negative_GE_TOLI_20091004_040040.jpg

✅ Finished! Open this directory in Finder to view the plots:
➡️  /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/outpu_error_sta_lta/error_analysis_plots
